<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-11-self-hosting/lesson-11.3-hybrid-litellm/notebooks/GCP_Capstone_11.3_HybridLiteLLM.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.3 Hybrid LiteLLM Gateway — Sensitive → Gemma, General → Gemini
**Netsetos GenAI Engineering — GCP Capstone**

Route intelligently based on PII classification. LiteLLM v1.83 unified API.


## Cell 1: LiteLLM config.yaml - Three Models, Tag Routing, Bidirectional Fallback


In [ ]:
CONFIG_YAML = '''
model_list:
  - model_name: documind-sensitive
    litellm_params:
      model: hosted_vllm/google/gemma-3-4b-it
      api_base: https://gemma-vllm-xxx-uc.a.run.app/v1
      api_key: "internal-token"
      rpm: 60
      tpm: 100000
      tags: ["sensitive", "hipaa", "gdpr", "dpdpa"]
      input_cost_per_token: 0.00001
      output_cost_per_token: 0.00001
  
  - model_name: documind-general
    litellm_params:
      model: vertex_ai/gemini-2.5-flash
      vertex_project: os.environ/VERTEXAI_PROJECT
      vertex_location: us-central1
      tags: ["general", "public"]
  
  - model_name: documind-reasoning
    litellm_params:
      model: vertex_ai/gemini-2.5-pro
      vertex_project: os.environ/VERTEXAI_PROJECT
      vertex_location: us-central1
      tags: ["complex", "reasoning"]

router_settings:
  routing_strategy: simple-shuffle
  enable_tag_filtering: true
  fallbacks:
    - documind-sensitive: ["documind-general"]
    - documind-general: ["documind-sensitive"]
    - documind-reasoning: ["documind-general"]
  num_retries: 3
  timeout: 30
  allowed_fails: 3
  cooldown_time: 60
  retry_policy:
    RateLimitErrorRetries: 3
    TimeoutErrorRetries: 2
    InternalServerErrorRetries: 2
    AuthenticationErrorRetries: 0

guardrails:
  - guardrail_name: "documind-pii-router"
    litellm_params:
      guardrail: custom
      mode: "pre_call"
      callback_class: documind_router.DocuMindRouter

litellm_settings:
  callbacks: ["langfuse_otel"]
  langfuse_default_tags: ["environment", "model", "tenant"]

general_settings:
  master_key: os.environ/LITELLM_MASTER_KEY
  database_url: os.environ/DATABASE_URL

tag_budget_config:
  tenant-acme:
    max_budget: 500.00
    budget_duration: "30d"
  tenant-enterprise:
    max_budget: 5000.00
    budget_duration: "30d"
'''
with open('config.yaml', 'w') as f:
    f.write(CONFIG_YAML)
print('config.yaml written')
print('Tag-based routing + bidirectional fallback + per-tenant budgets')


## Cell 2: PII Classifier — Regex + Presidio


In [ ]:
CLASSIFIER_PY = '''
import re
from enum import IntEnum
from presidio_analyzer import AnalyzerEngine

class SensitivityTier(IntEnum):
    PUBLIC = 0
    INTERNAL = 1
    CONFIDENTIAL = 2
    RESTRICTED = 3

PATTERNS_RESTRICTED = {
    "SSN": re.compile(r"\\b(?!000|666|9\\d{2})\\d{3}[-\\s]?\\d{2}[-\\s]?\\d{4}\\b"),
    "AADHAAR": re.compile(r"\\b[2-9]\\d{3}[-\\s]?\\d{4}[-\\s]?\\d{4}\\b"),
    "PAN": re.compile(r"\\b[A-Z]{5}[0-9]{4}[A-Z]\\b"),
    "CREDIT_CARD": re.compile(r"\\b(?:\\d[ -]*?){13,19}\\b"),
}
PATTERNS_CONFIDENTIAL = {
    "EMAIL": re.compile(r"\\b[\\w.-]+@[\\w.-]+\\.\\w+\\b"),
    "PHONE": re.compile(r"\\b\\d{3}[-.]?\\d{3}[-.]?\\d{4}\\b"),
}

_analyzer = None
def get_analyzer():
    global _analyzer
    if _analyzer is None:
        _analyzer = AnalyzerEngine()
    return _analyzer

def classify_tier(text: str) -> SensitivityTier:
    # Layer 1: fast regex for RESTRICTED
    for name, pattern in PATTERNS_RESTRICTED.items():
        if pattern.search(text):
            # Layer 2: Presidio confirms context-aware
            results = get_analyzer().analyze(text=text, language="en")
            if any(r.score >= 0.7 for r in results):
                return SensitivityTier.RESTRICTED
    # Layer 1 for CONFIDENTIAL
    for name, pattern in PATTERNS_CONFIDENTIAL.items():
        if pattern.search(text):
            return SensitivityTier.CONFIDENTIAL
    return SensitivityTier.PUBLIC

# Quick test
test_cases = [
    ("What is GDPR?", SensitivityTier.PUBLIC),
    ("Email me at user@acme.com", SensitivityTier.CONFIDENTIAL),
    ("My Aadhaar is 2345-6789-0123", SensitivityTier.RESTRICTED),
    ("SSN 123-45-6789", SensitivityTier.RESTRICTED),
    ("PAN ABCDE1234F", SensitivityTier.RESTRICTED),
]
'''
with open('documind_classifier.py', 'w') as f:
    f.write(CLASSIFIER_PY)
print('documind_classifier.py written')
print('Test cases: public, email, Aadhaar, SSN, PAN')
print('Expected classifications: PUBLIC, CONFIDENTIAL, RESTRICTED, RESTRICTED, RESTRICTED')


## Cell 3: LiteLLM CustomGuardrail — Pre-Call Hook That Routes


In [ ]:
ROUTER_PY = '''
from litellm.integrations.custom_guardrail import CustomGuardrail
from documind_classifier import classify_tier, SensitivityTier
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
import logging

logger = logging.getLogger("documind.router")

class DocuMindRouter(CustomGuardrail):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.analyzer = AnalyzerEngine()
        self.anonymizer = AnonymizerEngine()
    
    def mask_pii(self, data: dict) -> dict:
        """Replace PII with placeholders before sending to external API."""
        for msg in data.get("messages", []):
            content = msg.get("content")
            if not isinstance(content, str):
                continue
            results = self.analyzer.analyze(text=content, language="en")
            if results:
                anonymized = self.anonymizer.anonymize(text=content, analyzer_results=results)
                msg["content"] = anonymized.text
        return data
    
    async def async_pre_call_hook(self, user_api_key_dict, cache, data, call_type):
        """Runs BEFORE LiteLLM dispatches the request."""
        # Concatenate all user messages
        text = " ".join(
            m.get("content", "") for m in data.get("messages", [])
            if isinstance(m.get("content"), str)
        )
        
        tier = classify_tier(text)
        original_model = data.get("model")
        
        # Routing decision
        if tier == SensitivityTier.RESTRICTED:
            data["model"] = "documind-sensitive"
            logger.info(f"RESTRICTED routed to self-hosted (was {original_model})")
        elif tier == SensitivityTier.CONFIDENTIAL:
            data = self.mask_pii(data)
            data["model"] = "documind-general"
            logger.info(f"CONFIDENTIAL masked and routed to general")
        # else PUBLIC: keep original model selection
        
        # Log routing decision for audit trail
        data["metadata"] = data.get("metadata", {})
        data["metadata"]["routing_tier"] = tier.name
        data["metadata"]["original_model"] = original_model
        return data
'''
with open('documind_router.py', 'w') as f:
    f.write(ROUTER_PY)
print('documind_router.py written')
print('Pre-call hook modifies data["model"] based on classified tier')
print('Registered in config.yaml under guardrails: with mode: pre_call')


## Cell 4: Cloud DLP Async Audit Integration


In [ ]:
DLP_AUDIT_PY = '''
from google.cloud import dlp_v2, bigquery
import json, datetime, random, asyncio

dlp_client = dlp_v2.DlpServiceClient()
bq_client = bigquery.Client()
PROJECT_ID = "your-project-id"
DLP_TEMPLATE = f"projects/{PROJECT_ID}/locations/global/inspectTemplates/documind-pii"

def create_documind_inspect_template():
    """One-time setup: create inspect template with India info types."""
    template = {
        "inspect_config": {
            "info_types": [
                {"name": "EMAIL_ADDRESS"},
                {"name": "PHONE_NUMBER"},
                {"name": "CREDIT_CARD_NUMBER"},
                {"name": "US_SOCIAL_SECURITY_NUMBER"},
                {"name": "INDIA_AADHAAR_INDIVIDUAL"},
                {"name": "INDIA_PAN_INDIVIDUAL"},
                {"name": "INDIA_GST_INDIVIDUAL"},
                {"name": "PERSON_NAME"},
                {"name": "MEDICAL_TERM"},
                {"name": "DATE_OF_BIRTH"},
            ],
            "min_likelihood": dlp_v2.Likelihood.POSSIBLE,
            "include_quote": False,  # Don't include raw PII in findings
        },
        "display_name": "DocuMind PII Audit Template",
        "description": "Audit layer for hybrid routing validation",
    }
    return dlp_client.create_inspect_template(
        request={
            "parent": f"projects/{PROJECT_ID}/locations/global",
            "inspect_template": template,
            "template_id": "documind-pii"
        }
    )

async def async_audit_to_bigquery(request_id: str, text: str,
                                    classified_tier: str,
                                    sample_rate: float = 0.1):
    """Fire-and-forget DLP scan + BigQuery log.
    Only 10% of requests get DLP scanned (cost control).
    Findings compared against classified_tier to measure accuracy."""
    if random.random() > sample_rate:
        return
    
    try:
        # Scan with Cloud DLP
        response = await asyncio.to_thread(
            dlp_client.inspect_content,
            request={
                "parent": f"projects/{PROJECT_ID}/locations/global",
                "inspect_template_name": DLP_TEMPLATE,
                "item": {"value": text[:10000]}  # DLP has size limits
            }
        )
        
        findings = response.result.findings
        dlp_entities = [f.info_type.name for f in findings]
        
        # Determine DLP-inferred tier
        restricted_types = {"INDIA_AADHAAR_INDIVIDUAL", "INDIA_PAN_INDIVIDUAL",
                            "CREDIT_CARD_NUMBER", "US_SOCIAL_SECURITY_NUMBER",
                            "MEDICAL_TERM"}
        dlp_tier = ("RESTRICTED" if any(e in restricted_types for e in dlp_entities)
                    else "CONFIDENTIAL" if dlp_entities
                    else "PUBLIC")
        
        # Log for accuracy measurement
        bq_client.insert_rows_json(
            f"{PROJECT_ID}.documind.routing_audit",
            [{
                "request_id": request_id,
                "classified_tier": classified_tier,
                "dlp_tier": dlp_tier,
                "dlp_entities": dlp_entities,
                "match": classified_tier == dlp_tier,
                "timestamp": datetime.datetime.utcnow().isoformat(),
            }]
        )
    except Exception as e:
        logger.error(f"DLP audit failed: {e}")
'''
with open('dlp_audit.py', 'w') as f:
    f.write(DLP_AUDIT_PY)
print('dlp_audit.py written')
print('Async audit layer: 10% sample rate, $3/GB scanned')
print('BigQuery rollup reveals classifier accuracy against DLP ground truth')


## Cell 5: Dockerfile + Deploy LiteLLM to Cloud Run


In [ ]:
DOCKERFILE = '''
FROM docker.litellm.ai/berriai/litellm:main-stable

WORKDIR /app

# Install Presidio + Google Cloud SDKs for custom guardrail
RUN pip install --no-cache-dir \\
    presidio-analyzer==2.2.* \\
    presidio-anonymizer==2.2.* \\
    google-cloud-dlp>=3.24.0 \\
    google-cloud-bigquery>=3.25.0

# Download spaCy model for Presidio NER
RUN python -m spacy download en_core_web_lg

# Copy custom guardrail code
COPY documind_classifier.py documind_router.py dlp_audit.py ./

# config.yaml loads from GCS at runtime (LITELLM_CONFIG_BUCKET env)
ENV PORT=8080
EXPOSE 8080

CMD ["--config", "/app/config.yaml", "--port", "8080", "--host", "0.0.0.0"]
'''
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)

DEPLOY_CMD = '''
# Create config bucket
gsutil mb gs://$PROJECT_ID-litellm-config
gsutil cp config.yaml gs://$PROJECT_ID-litellm-config/config.yaml

# Create Postgres for state (Cloud SQL)
gcloud sql instances create litellm-db \\
  --database-version=POSTGRES_15 \\
  --cpu=1 --memory=4GB \\
  --region=us-central1 \\
  --network=default

# Deploy LiteLLM proxy to Cloud Run
gcloud run deploy litellm-gateway \\
  --source . \\
  --region us-central1 \\
  --cpu 1 --memory 2Gi \\
  --port 8080 \\
  --min-instances 1 \\
  --max-instances 5 \\
  --concurrency 100 \\
  --service-account litellm-sa@$PROJECT_ID.iam.gserviceaccount.com \\
  --set-env-vars="LITELLM_CONFIG_BUCKET_TYPE=gcs,LITELLM_CONFIG_BUCKET=$PROJECT_ID-litellm-config,VERTEXAI_PROJECT=$PROJECT_ID" \\
  --set-secrets="LITELLM_MASTER_KEY=litellm-master:latest,DATABASE_URL=litellm-db-url:latest" \\
  --vpc-egress=private-ranges-only \\
  --network=documind-vpc \\
  --subnet=documind-subnet

# Grant IAM permissions
gcloud projects add-iam-policy-binding $PROJECT_ID \\
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
  --role="roles/aiplatform.user"
gcloud projects add-iam-policy-binding $PROJECT_ID \\
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
  --role="roles/run.invoker"
gcloud projects add-iam-policy-binding $PROJECT_ID \\
  --member="serviceAccount:litellm-sa@$PROJECT_ID.iam.gserviceaccount.com" \\
  --role="roles/dlp.user"
'''
print('Dockerfile written')
print(DEPLOY_CMD)


## Cell 6: Client Code - Standard OpenAI SDK via LiteLLM Gateway


In [ ]:
CLIENT_PY = '''
from openai import OpenAI

GATEWAY_URL = "https://litellm-gateway-xxxxx-uc.a.run.app"
VIRTUAL_KEY = "sk-litellm-virtual-key-for-tenant-acme"

client = OpenAI(
    base_url=f"{GATEWAY_URL}/v1",
    api_key=VIRTUAL_KEY,
)

# CASE 1: General query -> Gemini Flash (automatic, no PII)
response = client.chat.completions.create(
    model="documind-general",
    messages=[{"role": "user", "content": "Summarize Q4 financial trends"}],
    extra_body={
        "metadata": {
            "tags": ["tenant-acme", "feature-summarization"],
            "user_id": "user-123",
        }
    }
)
print(f"Model: {response.model}")  # expected: gemini-2.5-flash
print(f"Tier: {response.metadata.get('routing_tier')}")  # expected: PUBLIC

# CASE 2: RESTRICTED query -> auto-routed to self-hosted Gemma
response = client.chat.completions.create(
    model="documind-general",  # Client requests general, router overrides to sensitive
    messages=[{
        "role": "user",
        "content": "Patient Aadhaar 2345-6789-0123 has diabetes. Summarize care plan."
    }],
    extra_body={
        "metadata": {
            "tags": ["tenant-acme", "feature-medical-records"],
        }
    }
)
print(f"Model: {response.model}")  # expected: gemma-3-4b-it (routed by PII detection)
print(f"Tier: {response.metadata.get('routing_tier')}")  # expected: RESTRICTED

# CASE 3: Complex reasoning -> Gemini Pro (explicit model selection)
response = client.chat.completions.create(
    model="documind-reasoning",
    messages=[{
        "role": "user",
        "content": "Analyze this 50-page contract for ambiguities and contradictions: ..."
    }],
    extra_body={
        "metadata": {
            "tags": ["tenant-acme", "feature-contract-analysis"],
        }
    }
)
print(f"Model: {response.model}")  # expected: gemini-2.5-pro

# Streaming works the same way
stream = client.chat.completions.create(
    model="documind-general",
    messages=[{"role": "user", "content": "Explain OAuth2"}],
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
'''
with open('client.py', 'w') as f:
    f.write(CLIENT_PY)
print('client.py written')
print('Standard OpenAI SDK unchanged. base_url points to LiteLLM gateway.')
print('Virtual keys enforce per-tenant budget caps + provide cost attribution.')


## Cell 7: Cost Dashboard SQL (BigQuery)


In [ ]:
DASHBOARD_SQL = '''
-- LiteLLM spend logs are automatically streamed to BigQuery
-- when general_settings.store_model_in_db: true is set in config.yaml

-- QUERY 1: Daily cost breakdown by model
SELECT
  DATE(start_time) AS date,
  model,
  COUNT(*) AS requests,
  SUM(total_tokens) AS tokens,
  ROUND(SUM(response_cost), 2) AS cost_usd
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30
GROUP BY date, model
ORDER BY date DESC, cost_usd DESC;

-- QUERY 2: Per-tenant spend (metadata tags)
SELECT
  JSON_VALUE(metadata, '$.tags[0]') AS tenant,
  model,
  COUNT(*) AS requests,
  ROUND(SUM(response_cost), 2) AS cost_usd
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30
GROUP BY tenant, model
ORDER BY cost_usd DESC;

-- QUERY 3: Routing tier distribution (compliance audit)
SELECT
  JSON_VALUE(metadata, '$.routing_tier') AS tier,
  model,
  COUNT(*) AS requests,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_of_traffic
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 7
GROUP BY tier, model
ORDER BY tier, requests DESC;

-- QUERY 4: ROUTING SAVINGS &mdash; counterfactual all-Pro cost
SELECT
  SUM(response_cost) AS actual_hybrid_cost,
  SUM(total_tokens * 0.000011) AS counterfactual_all_pro_cost,
  SUM(total_tokens * 0.000011) - SUM(response_cost) AS savings_usd,
  ROUND(100.0 * (1 - SUM(response_cost) / SUM(total_tokens * 0.000011)), 1) AS savings_pct
FROM `${PROJECT_ID}.litellm.spend_logs`
WHERE DATE(start_time) >= CURRENT_DATE() - 30;

-- QUERY 5: Classifier accuracy vs DLP ground truth
SELECT
  classified_tier,
  dlp_tier,
  COUNT(*) AS n,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY classified_tier), 1) AS pct
FROM `${PROJECT_ID}.documind.routing_audit`
WHERE DATE(timestamp) >= CURRENT_DATE() - 7
GROUP BY classified_tier, dlp_tier
ORDER BY classified_tier, n DESC;
'''
print(DASHBOARD_SQL)
print()
print('Typical DocuMind results:')
print('  actual_hybrid_cost: $1,113')
print('  counterfactual_all_pro: $7,932')
print('  savings_pct: 86%')
print()
print('Classifier accuracy target: >95% match with DLP for RESTRICTED tier')


## Cell 8: Shadow Mode + Enforcement Toggle


In [ ]:
# Shadow mode vs enforcement - the safe deployment pattern

SHADOW_CONFIG = '''
# config.yaml - SHADOW MODE (classify + log, don't enforce)
guardrails:
  - guardrail_name: "documind-pii-router"
    litellm_params:
      guardrail: custom
      mode: "pre_call"
      callback_class: documind_router.DocuMindRouter
      default_on: true
      # SHADOW MODE: log but do not actually re-route
      params:
        enforce: false   # ← KEY: classify + log, but don't modify data['model']
        audit_all: true  # Log every routing decision
'''

ENFORCE_CONFIG = '''
# config.yaml - PRODUCTION MODE (enforce routing)
guardrails:
  - guardrail_name: "documind-pii-router"
    litellm_params:
      guardrail: custom
      mode: "pre_call"
      callback_class: documind_router.DocuMindRouter
      default_on: true
      params:
        enforce: true     # ← flip to true after shadow mode validation
        audit_sample: 0.1 # DLP audit on 10% of requests
'''

print('DEPLOYMENT SEQUENCE:')
print('Week 1-2: Deploy with enforce: false')
print('  - Classifier runs on every request')
print('  - Routing decisions logged to BigQuery')
print('  - NO actual re-routing happens')
print('  - Review BigQuery accuracy report daily')
print()
print('Week 3: Deploy with enforce: true')
print('  - Routing now ACTIVELY redirects RESTRICTED to self-hosted')
print('  - Continue DLP audit on 10% sample')
print('  - Alert on classified_tier != dlp_tier for RESTRICTED')
print()
print('Week 4+: Monitor and tune')
print('  - Target: >99% precision for RESTRICTED tier')
print('  - Target: <50ms p95 classification latency')
print('  - Target: <5% over-classification (PUBLIC classified as CONFIDENTIAL)')


## Done!
Complete hybrid LLM routing system:
- LiteLLM v1.83 config.yaml with 3 models, tag routing, bidirectional fallbacks
- PII classifier: regex (Layer 1) + Presidio (Layer 2)
- CustomGuardrail pre-call hook that re-routes based on tier
- Cloud DLP async audit layer (10% sample, $3/GB)
- Dockerfile + Cloud Run deployment with VPC
- Standard OpenAI SDK client with virtual keys + tags
- BigQuery cost dashboard with counterfactual savings
- Shadow mode → enforcement deployment sequence
